In [6]:
import pandas as pd
import numpy as np
import pickle
import os
os.chdir("../../data")

In [ ]:
# Read preprocessed input df.
NA_VALUE = -99.0
ac_df = pd.read_csv('input_datasets/TCGA_LUAD_with_targets.csv', index_col=0)
gnn_df = pd.read_csv('input_datasets/TCGA_LUAD_with_targets_graph.tsv', sep='\t', index_col=0)
gnn_df.index = gnn_df.index.astype("str")
print(ac_df.isna().sum().sum())

In [ ]:
# Introduce simulated missingnes in non-NA values.
NA_RATIO = 0.5
NUM_RUNS = 10

for i in range(NUM_RUNS):
    # 1. Find all non-NA positions as (row_idx, col_name)
    df_sim = ac_df.copy()
    non_na_mask = df_sim.notna()
    non_na_positions = np.array(list(zip(*np.where(non_na_mask.values))))

    # Map positions to actual row and column labels
    row_indices = df_sim.index[non_na_positions[:, 0]]
    col_indices = df_sim.columns[non_na_positions[:, 1]]

    positions = list(zip(row_indices, col_indices))

    # 2. Choose given percentage of these positions to mask
    n_to_mask = max(1, int(NA_RATIO * len(positions)))  # at least 1
    masked_indices = np.random.choice(len(positions), size=n_to_mask, replace=False)
    groundtruth_values = dict()

    # 3. Mask those positions in the DataFrame by setting to NA
    for idx in masked_indices:
        r, c = positions[idx]
        groundtruth_values[(r, c)] = (df_sim.at[r, c], gnn_df.at[str(c), 'type'])

    print(f"Masked {len(groundtruth_values)} values at positions:")
    print(groundtruth_values)
    
    with open(f'imputation_groundtruth/TCGA_LUAD/masked_values_{NA_RATIO}_{i}.pkl', 'wb') as f:
        pickle.dump(groundtruth_values, f)


In [ ]:
# Generate copy-masked data for graph-based imputation.
directory = 'imputation_groundtruth/TCGA_LUAD/'
out_directory = 'imputation_data/pome_based/TCGA_LUAD/simulated_data'
NA_VALUE = -99.0

# Iterate over all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.pkl'):
        # Full file path
        file_path = os.path.join(directory, filename)
        
        # Load pickle file
        with open(file_path, 'rb') as f:
            dict_data = pickle.load(f)
        
        # Get filename without extension
        name_without_ext = os.path.splitext(filename)[0]
        
        # Split by underscores
        parts = name_without_ext.split('_')
        
        # Extract simulated NA ration and number of run.
        na_ratio = float(parts[2])
        num_run = float(parts[3])
        
        # Copy-mask data in input df.
        gnn_sim = gnn_df.copy()
        for pos, _ in dict_data.items():
            gnn_sim.loc[pos[1], pos[0]] = NA_VALUE
        
        output_file = os.path.join(out_directory, f'masked_values_{na_ratio}_{num_run}.tsv')
        gnn_sim.to_csv(output_file, sep='\t', index=True)

            

In [ ]:
# Generate copy-masked data for AutoComplete imputation.
directory = 'imputation_groundtruth/TCGA_LUAD/'
out_directory = 'imputation_data/autocomplete/TCGA_LUAD/simulated_data'
NA_VALUE = pd.NA

# Iterate over all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.pkl'):
        # Full file path
        file_path = os.path.join(directory, filename)
        
        # Load pickle file
        with open(file_path, 'rb') as f:
            dict_data = pickle.load(f)
        
        # Get filename without extension
        name_without_ext = os.path.splitext(filename)[0]
        
        # Split by underscores
        parts = name_without_ext.split('_')
        
        # Extract simulated NA ration and number of run.
        na_ratio = float(parts[2])
        num_run = float(parts[3])
        
        # Copy-mask data in input df.
        ac_sim = ac_df.copy()
        for pos, _ in dict_data.items():
            ac_sim.loc[pos[0], pos[1]] = NA_VALUE
        
        output_file = os.path.join(out_directory, f'masked_values_{na_ratio}_{num_run}.tsv')
        ac_sim.to_csv(output_file, sep='\t', index=True)
